# Translation ambiguity in nanobeam ptychography

Each reconstruction is registered against the ground-truth potential; the recorded shift is one
row of a `shifts_conv*_{kind}.csv`. When the diffracted disks do **not** overlap the reconstruction
is only determined up to a translation within the projected unit cell, so the shift is a random
vector uniformly distributed over that cell. Once the disks overlap the solution is unique and the
shift vanishes.

The overlap threshold depends on the zone axis, $\alpha_c = \lambda / 2d$ for the first allowed
ZOLZ reflection:

| particle | zone | reflection | $d$ | $\alpha_c$ |
|---|---|---|---|---|
| single NP, two-NP upper | [111] | {220} | 1.442 Å | **6.8 mrad** |
| two-NP lower | [100] | {200} | 2.039 Å | **4.8 mrad** |

so the two particles in the *same* reconstruction are predicted to become unique at different
convergence angles. All analysis lives in [`ambiguity_analysis.py`](ambiguity_analysis.py).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import ambiguity_analysis as sa

plt.rcParams.update({
    "font.size": 9, "axes.titlesize": 9, "figure.dpi": 140,
    "savefig.bbox": "tight", "text.color": sa.INK, "axes.labelcolor": sa.INK_MUTED,
})

# (sample, on amorphous carbon, row label) -- one row of the primary figure each
ROWS = [
    ("single_np", False, "single NP\n[111], vacuum"),
    ("single_np", True,  "single NP\n[111], on aC"),
    ("two_upper", False, "two NPs, upper\n[111]"),
    ("two_lower", False, "two NPs, lower\n[100]"),
]
ANGLES = sa.CONVERGENCE_ANGLES

data, meta = {}, {}
for key, on_aC, _ in ROWS:
    runs, offset, scale = sa.build(sa.SAMPLES[key], on_aC)
    data[(key, on_aC)] = runs
    meta[(key, on_aC)] = (offset, scale)

## Calibration

Two numbers per dataset used to be hand-tuned in the old notebooks; both are measured here.

**Registration offset.** At 12 mrad the solution is unique, so whatever shift survives is
instrumental. `calibrate_offset` takes it from that run and refuses to proceed if the run is not
in fact deterministic. (These reproduce the old hard-coded `correction_x/correction_y`.)

**Cell scale.** The old `4.2 px` per unit cell drifts between cells (4.0 / 4.18 / 4.2) and cannot
be right for both fields of view. `fit_cell_scale` fixes it per dataset by matching the observed
median radius of the fully non-unique runs to the null's, which is a fixed fraction of the cell.

In [ ]:
rows = []
for key, on_aC, _ in ROWS:
    s = sa.SAMPLES[key]
    offset, scale = meta[(key, on_aC)]
    _, spread = sa.calibrate_offset(s, on_aC)
    rows.append({
        "sample": s.label, "on_aC": on_aC,
        "offset_x (px)": offset[0], "offset_y (px)": offset[1],
        "12 mrad p95 spread (px)": spread,
        "cell scale (px)": scale, "legacy (px)": 4.2,
        "alpha_c (mrad)": s.alpha_critical,
    })
pd.DataFrame(rows).round(3)

### Two independent checks on the cell scale

The fit above assumes the 1.5 and 3 mrad runs are *fully* non-unique. Two things test that
without making the assumption:

1. the lattice period measured straight off a reconstruction image (single-particle images only —
   the two-particle field of view mixes both orientations);
2. the two particles in the two-NP simulation share a pixel size, so their fitted scales must be
   in the ratio $d_{\rm nn}[111] / d_{\rm nn}[100] = \sqrt2$. With only ~30 seeds at 3 mrad this
   is the noisier of the two checks.

In [ ]:
import glob

for key, on_aC in [("single_np", False), ("single_np", True)]:
    pngs = [p for a in ANGLES
            for p in sorted(glob.glob(f"{sa.run_dir(sa.SAMPLES[key], a, on_aC)}/reconstruction_*.png"))]
    if not pngs:
        print(f"single NP, on_aC={on_aC}: no reconstruction image saved for this run")
        continue
    measured = sa.estimate_cell_scale_from_png(pngs[0])
    fitted = meta[(key, on_aC)][1]
    print(f"single NP, on_aC={on_aC!s:5s}  fitted {fitted:.2f} px | image FFT {measured:.2f} px"
          f" | legacy 4.20 px  ->  fit vs image {abs(measured - fitted) / measured:.1%}")

ratio = meta[("two_upper", False)][1] / meta[("two_lower", False)][1]
print(f"\ntwo NPs, upper/lower scale ratio {ratio:.2f} (expected {np.sqrt(2):.2f})")

## Figure 1 — where the reconstruction lands inside the unit cell

The primary result. Each point is one random seed, folded into the Wigner–Seitz cell of that
particle's projected lattice. Left to right the cell fills uniformly and then collapses onto the
true column position; the [100] row collapses one column earlier than the [111] rows, exactly
where its own disk-overlap threshold sits. Runs with fewer than five seeds are left blank -- one
dot is not a distribution.

In [ ]:
fig, axes = plt.subplots(len(ROWS), len(ANGLES), figsize=(7.2, 7.6))

for i, (key, on_aC, label) in enumerate(ROWS):
    s = sa.SAMPLES[key]
    runs = data[(key, on_aC)]
    for j, angle in enumerate(ANGLES):
        ax = axes[i, j]
        df = runs.get(angle, pd.DataFrame(columns=["dx", "dy", "r"]))
        sa.plot_cell_scatter(ax, df, s, sa.ANGLE_COLORS[angle],
                             disks_overlap=angle >= s.alpha_critical)
        if i == 0:
            ax.set_title(f"{angle:g} mrad", color=sa.INK, pad=6)
        if j == 0:
            ax.text(-0.16, 0.5, label, transform=ax.transAxes, rotation=90,
                    ha="center", va="center", fontsize=8.5, color=sa.INK)

fig.subplots_adjust(top=0.86, hspace=0.12, wspace=0.06)
fig.suptitle("Reconstruction shift within the projected unit cell", y=0.975,
             fontsize=11, color=sa.INK)
fig.text(0.5, 0.928, "one point per random seed;  outline = Wigner-Seitz cell,  "
         "+ = true atom column;  % unique = seeds landing on the true solution\n"
         "shaded panels = convergence angle above that particle's disk-overlap "
         "threshold $\\alpha_c = \\lambda / 2d$",
         ha="center", va="top", fontsize=7.8, color=sa.INK_MUTED)

for ext in ("pdf", "png"):
    fig.savefig(f"fig1_shift_cell_grid.{ext}")

## Figure 2 — magnitude distribution and the uniqueness transition

(a) ECDFs rather than histograms: no binning, and comparable across runs whose N differs by 20×.
The 1.5 and 3 mrad curves sit on the uniform-over-cell null, 6 mrad departs from it, 12 mrad is a
step at zero.

(b) The transition lands at each particle's own $\alpha_c$.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.6, 3.2))

sa.plot_ecdf(ax1, data[("single_np", False)], sa.SAMPLES["single_np"])
ax1.set_xlabel("|shift|  (nearest-column spacing)")
ax1.set_ylabel("fraction of seeds")
ax1.set_title("single NP [111], vacuum", color=sa.INK, loc="left")
ax1.legend(frameon=False, fontsize=7.5, loc="upper left", labelcolor=sa.INK_MUTED,
           borderaxespad=0.6, handlelength=1.6, labelspacing=0.35)

for (key, on_aC, label), marker in zip(ROWS, ["o", "s", "^", "D"]):
    s = sa.SAMPLES[key]
    runs = data[(key, on_aC)]
    xs = [a for a in ANGLES if a in runs and len(runs[a]) >= 5]
    ys = [(runs[a]["r"] < 0.05).mean() for a in xs]
    ax2.plot(xs, ys, marker=marker, ms=5, lw=1.6,
             color="#1c5cab" if s.zone == "111" else "#eb6834",
             ls="--" if on_aC else "-", label=label.replace("\n", " "))

for zone, color, ha in [("100", "#eb6834", "right"), ("111", "#1c5cab", "left")]:
    a_c = next(s.alpha_critical for s in sa.SAMPLES.values() if s.zone == zone)
    ax2.axvline(a_c, color=color, lw=1.0, ls=":", zorder=0)
    ax2.text(a_c, 0.55, f"  $\\alpha_c$[{zone}] = {a_c:.1f} mrad  ", color=color,
             fontsize=7, ha=ha, va="center", rotation=90,
             bbox=dict(fc="white", ec="none", pad=0.5))

ax2.set_xscale("log")
ax2.set_xticks(ANGLES)
ax2.set_xticklabels([f"{a:g}" for a in ANGLES])
ax2.minorticks_off()
ax2.set_xlabel("convergence semi-angle (mrad)")
ax2.set_ylabel("fraction of seeds at\nthe true solution")
ax2.set_ylim(-0.05, 1.08)
ax2.legend(frameon=False, fontsize=7, loc="upper left", labelcolor=sa.INK_MUTED)
sa.style_axes(ax2)
fig.subplots_adjust(wspace=0.32)

for ext in ("pdf", "png"):
    fig.savefig(f"fig2_magnitude_and_transition.{ext}")

## Figure 3 — direction carries no information (supplementary)

The ambiguity has no preferred direction: the Rayleigh test never rejects uniformity. At 12 mrad
the shift is zero, so its direction is undefined rather than uniform — which is why this is a
supporting panel and not the main figure.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(7.6, 2.3), subplot_kw={"projection": "polar"})

runs = data[("single_np", False)]
for ax, angle in zip(axes, ANGLES):
    df = runs[angle]
    sa.plot_rose(ax, df, sa.ANGLE_COLORS[angle])
    n, z, p = sa.rayleigh_test(df["theta_deg"], df["r"])
    note = f"Rayleigh p = {p:.2f}" if np.isfinite(p) else "no direction (unique)"
    ax.set_title(f"{angle:g} mrad\n{note}", fontsize=8, color=sa.INK, pad=8)

fig.subplots_adjust(top=0.62, wspace=0.55)
fig.text(0.5, 0.99, "Shift direction is uniformly distributed at every convergence angle",
         ha="center", va="top", fontsize=9.5, color=sa.INK)
fig.text(0.5, 0.90, "single NP [111], vacuum;  dashed circle = uniform expectation",
         ha="center", va="top", fontsize=7.5, color=sa.INK_MUTED)

for ext in ("pdf", "png"):
    fig.savefig(f"fig3_direction_rose.{ext}")

## Summary table

`unique_frac` is the fraction of seeds within 0.05 cells of the true solution; `ks_vs_null` is the
distance from the fully-non-unique null (small = indistinguishable from complete ambiguity,
1 = perfectly unique); `wrapped_frac` is the fraction of rows that registration had locked onto a
neighbouring lattice peak, folded back into the cell here.

In [ ]:
summary = pd.DataFrame([
    sa.summarize(df, sa.SAMPLES[key])
    for key, on_aC, _ in ROWS
    for df in data[(key, on_aC)].values()
])
summary.to_csv("ambiguity_summary.csv", index=False)
summary.round(3)